# 85점 코드 기반 H100 SOTA 파이프라인

### 85점 코드에서 잘한 것 (유지)
- **Train + Dev 합쳐서 학습** (데이터 양 ↑)
- **영어 시스템 프롬프트** (Qwen은 영어 이해도가 더 높음)
- **선택지 셔플 안 함** (셔플이 오히려 혼란 줌)
- **토큰 변형 매칭** (a/A/ a/ A 전부 커버)
- **간결한 DataCollator**

### H100으로 올리면서 바꿀 것
- 3B → **7B** (H100 80GB니까)
- fp16 → **bf16 + SDPA**
- IMAGE_SIZE 384 → **512**
- BS=1 → **BS=4**, GRAD_ACCUM=4→**8** (유효 32)
- Epoch 3 → **5**
- LoRA r=16 → **r=64**
- 추론에 **TTA 8순열 Softmax** 추가
- 에폭별 **확률 CSV 저장** + **Soft Voting 앙상블**
- dev.csv **majority vote** 그대로 활용

## 1. 환경 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "Pillow>=10.0,<12.0"
!pip install -q "transformers>=4.45.0,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" \
    datasets pandas --upgrade

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("✅ 환경 준비 완료")

## 2. 설정 & 데이터 로드

In [ ]:
import os, re, math, random
from collections import Counter
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda"
COMPUTE_DTYPE = torch.bfloat16

# ── 설정값 ────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen2.5-VL-7B-Instruct"  # 3B→7B 상향
IMAGE_SIZE  = 512       # 384→512 상향
BATCH_SIZE  = 4         # 1→4 상향
GRAD_ACCUM  = 8         # 유효 배치 = 32
EPOCHS      = 5         # 3→5 상향
LR          = 2e-5      # 7B에 맞게 낮춤
WARMUP_RATIO = 0.05
PATIENCE    = 3
SEED        = 42

# ── 경로 ──────────────────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/vqa_h100_v2"
DATA_BASE  = "/content/drive/MyDrive/2026-ssafy-15-2-ai_dataset/2026-ssafy-15-2-ai_dataset"
BEST_DIR   = os.path.join(DRIVE_BASE, "best_model")
PROBS_DIR  = os.path.join(DRIVE_BASE, "probs")
SUB_DIR    = os.path.join(DRIVE_BASE, "submissions")
for d in [BEST_DIR, PROBS_DIR, SUB_DIR]:
    os.makedirs(d, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── 데이터 로드 (85점 코드 핵심: Train + Dev 합치기) ─────
def majority_vote(row):
    votes = [row[f"answer{i}"] for i in range(1, 6)]
    cnt = Counter(votes)
    return cnt.most_common(1)[0][0]

train_df = pd.read_csv(os.path.join(DATA_BASE, "train.csv"))
train_df["path"] = train_df["path"].apply(lambda p: os.path.join(DATA_BASE, p) if not p.startswith("/") else p)
train_df = train_df.dropna(subset=["answer"]).reset_index(drop=True)

dev_df = pd.read_csv(os.path.join(DATA_BASE, "dev.csv"))
dev_df["answer"] = dev_df.apply(majority_vote, axis=1)
dev_df["path"] = dev_df["path"].apply(lambda p: os.path.join(DATA_BASE, p) if not p.startswith("/") else p)

test_df = pd.read_csv(os.path.join(DATA_BASE, "test.csv"))
test_df["path"] = test_df["path"].apply(lambda p: os.path.join(DATA_BASE, p) if not p.startswith("/") else p)

# Train + Dev 합치기 (85점 코드의 핵심 전략)
all_df = pd.concat([train_df, dev_df], ignore_index=True)

print(f"Train: {len(train_df):,}개")
print(f"Dev  : {len(dev_df):,}개 (majority vote)")
print(f"합계 : {len(all_df):,}개")
print(f"Test : {len(test_df):,}개")
print(f"정답 분포:\n{all_df['answer'].value_counts().sort_index()}")
print("✅ 데이터 로드 완료")

## 3. 프롬프트 (영어 — 85점 코드 유지)

In [ ]:
# 85점 코드의 핵심: 영어 프롬프트 (Qwen은 영어 이해도가 더 높음)
SYSTEM_INSTRUCT = (
    "You are an expert visual question answering assistant specializing in "
    "recycling and waste classification. "
    "You carefully analyze the image and select the single best answer "
    "from the given options. "
    "Output ONLY one lowercase letter: a, b, c, or d. "
    "No explanation, no punctuation."
)

def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Answer with a single lowercase letter: a, b, c, or d."
    )

print("✅ 프롬프트 준비 완료")

## 4. 모델 & Processor 로드

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# 토큰 변형 매핑 (85점 코드의 핵심: 대소문자+공백 변형 전부 커버)
CHOICE_VARIANTS = {
    "a": ["a", "A", " a", " A"],
    "b": ["b", "B", " b", " B"],
    "c": ["c", "C", " c", " C"],
    "d": ["d", "D", " d", " D"],
}

def build_token_id_map(tokenizer, variants):
    token_map = {}
    for choice, texts in variants.items():
        ids = set()
        for text in texts:
            encoded = tokenizer.encode(text, add_special_tokens=False)
            if len(encoded) == 1:
                ids.add(encoded[0])
        token_map[choice] = list(ids)
    return token_map

CHOICE_TOKEN_IDS = build_token_id_map(processor.tokenizer, CHOICE_VARIANTS)
print("토큰 매핑:", {k: v for k, v in CHOICE_TOKEN_IDS.items()})

# 모델 로드 — bf16 + SDPA (H100 최적화)
print(f"\n모델 로딩: {MODEL_ID}")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
    device_map="auto",
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# LoRA (r=16→64 상향)
lora_config = LoraConfig(
    r=64, lora_alpha=128,
    lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("✅ 모델 로드 완료")

## 5. Dataset & DataCollator (85점 방식 유지 — 셔플 없음)

In [ ]:
class VQAMCDataset(Dataset):
    # 85점 코드 핵심: 선택지 셔플 안 함 (셔플이 오히려 성능 떨어뜨림)
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]),
            str(row["c"]), str(row["d"]))

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text}
            ]}
        ]

        if self.train:
            messages.append(
                {"role": "assistant", "content": [
                    {"type": "text", "text": str(row["answer"])}
                ]}
            )

        return {"messages": messages, "image": img,
                "answer": str(row["answer"]) if "answer" in row.index else None}


@dataclass
class DataCollator:
    # 85점 코드 방식: 간결한 collator
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for item in batch:
            text = self.processor.apply_chat_template(
                item["messages"], tokenize=False,
                add_generation_prompt=not self.train)
            texts.append(text)
            images.append([item["image"]])

        inputs = self.processor(
            text=texts, images=images,
            return_tensors="pt", padding=True)

        if self.train:
            labels = inputs["input_ids"].clone()
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            inputs["labels"] = labels

        if not self.train:
            answers = [item["answer"] for item in batch]
            return dict(inputs), answers

        return dict(inputs)

print("✅ Dataset & Collator 준비 완료")

## 6. DataLoader (Train+Dev 합쳐서)

In [ ]:
# 85점 핵심: Train + Dev 합쳐서 학습 데이터 양 늘리기
all_df_shuffled = all_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split_idx    = int(len(all_df_shuffled) * 0.9)
train_subset = all_df_shuffled.iloc[:split_idx].reset_index(drop=True)
valid_subset = all_df_shuffled.iloc[split_idx:].reset_index(drop=True)

print(f"학습: {len(train_subset):,}개 | 검증: {len(valid_subset):,}개")

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, train=True),
    num_workers=2, pin_memory=True)

valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, train=True),
    num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader):,} | Valid batches: {len(valid_loader):,}")
print(f"유효 배치: {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")
print("✅ DataLoader 준비 완료")

## 7. Optimizer & Scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.01)

num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps   = int(num_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps, num_training_steps)

print(f"총 스텝: {num_training_steps:,} | Warmup: {num_warmup_steps:,}")
print("✅ Optimizer & Scheduler 준비 완료")

## 8. 에폭별 Test 확률 추출 함수

에폭마다 test 데이터의 a/b/c/d softmax 확률을 CSV로 저장.

In [ ]:
def extract_test_probs(model, test_df, processor, token_ids, device, epoch_num):
    model.eval()
    all_probs = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc=f"Epoch {epoch_num} test probs"):
        img = Image.open(row["path"]).convert("RGB")
        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"]))

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text}]}
        ]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        enc = processor(text=[text], images=[[img]], return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**enc)

        last_logits = outputs.logits[0, -1, :]

        # 토큰 변형 매핑 방식 (85점 코드)
        scores = {}
        for choice, tids in token_ids.items():
            if tids:
                scores[choice] = last_logits[tids].max().item()
            else:
                scores[choice] = float("-inf")

        # softmax 확률 변환
        score_tensor = torch.tensor([scores["a"], scores["b"], scores["c"], scores["d"]])
        probs = F.softmax(score_tensor, dim=-1).numpy()

        all_probs.append({
            "id": row["id"],
            "prob_a": probs[0], "prob_b": probs[1],
            "prob_c": probs[2], "prob_d": probs[3],
        })

    probs_df = pd.DataFrame(all_probs)
    save_path = os.path.join(PROBS_DIR, f"probs_epoch_{epoch_num}.csv")
    probs_df.to_csv(save_path, index=False)
    print(f"✅ 저장: {save_path}")
    return probs_df

print("✅ 확률 추출 함수 준비 완료")

## 9. Fine-tuning 루프

In [ ]:
best_val_acc = 0.0
global_step  = 0
patience_counter = 0
optimizer.zero_grad(set_to_none=True)

print("=" * 60)
print(f"Fine-tuning 시작 | {EPOCHS} epochs | BS={BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")
print("=" * 60)

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────
    model.train()
    running_loss, num_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]",
                unit="batch", dynamic_ncols=True)

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**batch)
            raw_loss = outputs.loss

            if not torch.isfinite(raw_loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            loss = raw_loss / GRAD_ACCUM

        loss.backward()
        running_loss += raw_loss.item()
        num_batches += 1

        if (step % GRAD_ACCUM == 0) or (step == len(train_loader)):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        pbar.set_postfix({
            "loss": f"{running_loss/max(num_batches,1):.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
        })

    avg_train_loss = running_loss / max(num_batches, 1)

    # ── Validation (logit 방식 — 85점 코드) ───────────────
    model.eval()
    correct, total = 0, 0

    for row in tqdm(valid_subset.itertuples(), total=len(valid_subset),
                    desc=f"Epoch {epoch+1} [valid]"):
        img = Image.open(row.path).convert("RGB")

        user_text = build_mc_prompt(row.question, row.a, row.b, row.c, row.d)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text}
            ]}
        ]
        text_input = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[text_input], images=[[img]],
            return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**inputs)

        last_logits = outputs.logits[0, -1, :]
        scores = {}
        for choice, tids in CHOICE_TOKEN_IDS.items():
            if tids:
                scores[choice] = last_logits[tids].max().item()
            else:
                scores[choice] = float("-inf")

        pred = max(scores, key=scores.get)
        if pred == row.answer:
            correct += 1
        total += 1

    val_acc = correct / max(total, 1)
    print(f"\n[Epoch {epoch+1}] train_loss={avg_train_loss:.4f} | "
          f"val_acc={val_acc:.4f} ({correct}/{total})")
    print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

    # ── 에폭별 Test 확률 저장 ─────────────────────────────
    extract_test_probs(model, test_df, processor, CHOICE_TOKEN_IDS, device, epoch+1)

    # ── Best 모델 저장 ────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)
        print(f"  ✅ Best 저장 (val_acc={best_val_acc:.4f})")
    else:
        patience_counter += 1
        print(f"  ⚠️ 개선 없음 ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("🛑 Early Stopping!")
            break

print(f"\n학습 완료 — Best val_acc: {best_val_acc:.4f}")

## 10. TTA Inference (Softmax Soft Voting)

Best 모델로 8순열 TTA 추론.

In [ ]:
import gc
del model, base_model
gc.collect()
torch.cuda.empty_cache()

# Best 모델 로드
inf_base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa", device_map="auto", trust_remote_code=True)
inf_model = PeftModel.from_pretrained(inf_base, BEST_DIR)
inf_model.eval()

inf_processor = AutoProcessor.from_pretrained(
    BEST_DIR, min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE, trust_remote_code=True)

INF_TOKEN_IDS = build_token_id_map(inf_processor.tokenizer, CHOICE_VARIANTS)

TTA_PERMS = [
    [0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0],
    [0,2,1,3],[1,3,0,2],[2,0,3,1],[3,1,2,0],
]

def predict_tta(model, row, processor, token_ids, device):
    img = Image.open(row["path"]).convert("RGB")
    options = [("a",str(row["a"])),("b",str(row["b"])),
               ("c",str(row["c"])),("d",str(row["d"]))]
    vote_probs = {"a":0.0,"b":0.0,"c":0.0,"d":0.0}

    for perm in TTA_PERMS:
        reordered = [options[i] for i in perm]
        remap = {}
        for nl, (ol, _) in zip(["a","b","c","d"], reordered):
            remap[nl] = ol

        user_text = build_mc_prompt(
            str(row["question"]),
            reordered[0][1], reordered[1][1],
            reordered[2][1], reordered[3][1])

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}]}
        ]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        enc = processor(text=[text], images=[[img]], return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**enc)

        last_logits = outputs.logits[0, -1, :]
        scores = {}
        for ch, tids in token_ids.items():
            scores[ch] = last_logits[tids].max().item() if tids else float("-inf")

        st = torch.tensor([scores["a"],scores["b"],scores["c"],scores["d"]])
        probs = F.softmax(st, dim=-1).numpy()
        prob_dict = {"a":probs[0],"b":probs[1],"c":probs[2],"d":probs[3]}

        for nl, p in prob_dict.items():
            vote_probs[remap[nl]] += p

    for k in vote_probs:
        vote_probs[k] /= len(TTA_PERMS)
    return max(vote_probs, key=vote_probs.get), vote_probs

# 실행
print(f"TTA: {len(test_df):,}샘플 × {len(TTA_PERMS)}순열")
tta_preds, tta_probs_list = [], []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df),
                     desc="TTA Inference", dynamic_ncols=True):
    pred, probs = predict_tta(inf_model, row, inf_processor, INF_TOKEN_IDS, device)
    tta_preds.append(pred)
    tta_probs_list.append({"id":row["id"],
        "prob_a":probs["a"],"prob_b":probs["b"],
        "prob_c":probs["c"],"prob_d":probs["d"]})

    if (idx+1) % 500 == 0:
        pd.DataFrame(tta_probs_list).to_csv(
            os.path.join(PROBS_DIR,"probs_tta_checkpoint.csv"), index=False)

tta_probs_df = pd.DataFrame(tta_probs_list)
tta_probs_df.to_csv(os.path.join(PROBS_DIR,"probs_tta_best.csv"), index=False)

tta_sub = pd.DataFrame({"id":test_df["id"],"answer":tta_preds})
tta_sub.to_csv(os.path.join(SUB_DIR,"submission_tta.csv"), index=False)
print(f"✅ TTA 제출 저장 완료")
print(f"분포: {tta_sub['answer'].value_counts().sort_index().to_dict()}")

## 11. 멀티에폭 Soft Voting 앙상블 → 최종 제출

In [ ]:
print("=" * 60)
print("Soft Voting 앙상블")
print("=" * 60)

available_epochs = []
for f in sorted(os.listdir(PROBS_DIR)):
    if f.startswith("probs_epoch_") and f.endswith(".csv"):
        ep = int(f.replace("probs_epoch_","").replace(".csv",""))
        available_epochs.append(ep)

ensemble_epochs = available_epochs[-3:]
print(f"앙상블 에폭: {ensemble_epochs}")

label_map = {0:"a",1:"b",2:"c",3:"d"}

if len(ensemble_epochs) >= 2:
    prob_dfs = []
    for ep in ensemble_epochs:
        df = pd.read_csv(os.path.join(PROBS_DIR, f"probs_epoch_{ep}.csv"))
        prob_dfs.append(df)

    ens_df = prob_dfs[0][["id"]].copy()
    for col in ["prob_a","prob_b","prob_c","prob_d"]:
        ens_df[col] = np.mean([df[col].values for df in prob_dfs], axis=0)

    # TTA 50% + 에폭앙상블 50%
    final_df = tta_probs_df[["id"]].copy()
    for col in ["prob_a","prob_b","prob_c","prob_d"]:
        final_df[col] = 0.5 * tta_probs_df[col].values + 0.5 * ens_df[col].values
else:
    final_df = tta_probs_df.copy()

fm = final_df[["prob_a","prob_b","prob_c","prob_d"]].values
final_preds = [label_map[i] for i in fm.argmax(axis=1)]

final_sub = pd.DataFrame({"id":final_df["id"], "answer":final_preds})
final_path = os.path.join(SUB_DIR, "geun_sub.csv")
final_sub.to_csv(final_path, index=False)

assert len(final_sub) == len(test_df)
assert final_sub["answer"].isin(["a","b","c","d"]).all()

print(f"\n🏆 최종 제출: {final_path}")
print(f"분포: {final_sub['answer'].value_counts().sort_index().to_dict()}")
print(final_sub.head(10).to_string(index=False))
print("\n✅ 완료! 🎉")